### **Import Packages**

In [ ]:
import requests
from bs4 import BeautifulSoup
import os, time
from urllib.parse import urljoin
import pdfplumber
from pathlib import Path
import pdfplumber
import re
import pandas as pd



In [7]:


url = "https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates"
resp = requests.get(url, timeout=30)
print("Status:", resp.status_code)
print(resp.text[:1000])   


Status: 200
<!DOCTYPE html>
<html lang="en-gb" dir="ltr">
	<head>
		<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=3.0, user-scalable=yes"/>
		<meta name="HandheldFriendly" content="true" />
		<meta name="apple-mobile-web-app-capable" content="YES" />
		<meta charset="utf-8" />
	<base href="https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates" />
	<meta name="description" content="Central Bank Site" />
	<meta name="generator" content="Joomla! - Open Source Content Management" />
	<title>Daily Exchange Rates</title>
	<link href="/index.php/research/markets/exchange-rates/13-daily-exchange-rates?format=feed&amp;type=rss" rel="alternate" type="application/rss+xml" title="RSS 2.0" />
	<link href="/index.php/research/markets/exchange-rates/13-daily-exchange-rates?format=feed&amp;type=atom" rel="alternate" type="application/atom+xml" title="Atom 1.0" />
	<link href="/templates/beez3/favicon.ico" rel="shortcut icon" type=

In [8]:
SAVE_DIR = "rbz_pdfs_may_sep_2025"
os.makedirs(SAVE_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

MONTH_PAGES = [
    "https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates/1484-may-2025",
    "https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates/1467-june-2025",
    "https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates/1466-july-2025",
    "https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates/1465-august-2025",
    "https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates/1479-september-2025"
]

def get(url):
    r = session.get(url, timeout=40)
    r.raise_for_status()
    return r

def parse_month_page(url):
    html = get(url).text
    soup = BeautifulSoup(html, "html.parser")
    pdfs = []
    for a in soup.find_all("a", href=True):
        if a["href"].lower().endswith(".pdf"):
            pdfs.append(urljoin(url, a["href"]))
    return pdfs

def download_all():
    seen = set()
    for url in MONTH_PAGES:
        print(f"\nScraping {url}")
        pdfs = parse_month_page(url)
        print(f"  Found {len(pdfs)} PDFs")
        for pdf_url in pdfs:
            fname = os.path.basename(pdf_url).split("?")[0]
            path = os.path.join(SAVE_DIR, fname)
            if pdf_url not in seen and not os.path.exists(path):
                try:
                    resp = get(pdf_url)
                    with open(path, "wb") as f:
                        f.write(resp.content)
                    print(f"    ✓ Saved: {path}")
                    seen.add(pdf_url)
                except Exception as e:
                    print(f"    ✗ Error downloading {pdf_url}: {e}")
                time.sleep(1)

if __name__ == "__main__":
    download_all()



Scraping https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates/1484-may-2025
  Found 20 PDFs

Scraping https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates/1467-june-2025
  Found 21 PDFs

Scraping https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates/1466-july-2025
  Found 23 PDFs

Scraping https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates/1465-august-2025
  Found 19 PDFs

Scraping https://www.rbz.co.zw/index.php/research/markets/exchange-rates/13-daily-exchange-rates/1479-september-2025
  Found 3 PDFs


In [9]:
pdf_path = Path("rbz_pdfs_may_sep_2025/RATES_01_AUGUST_2025.pdf")

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()
        print(f"\n--- Page {i} ---\n")
        print(text[:1000])  


CropBox missing from /Page, defaulting to MediaBox


CropBox missing from /Page, defaulting to MediaBox



--- Page 1 ---

Friday, 01 August 2025
CURRENCY INDICES BID ASK MID RATE BID RATE ASK RATE MID RATE
ZWG ZWG ZWG
INTERBANK RATE
USD 1 1 1 .0000 26.1208 27.4604 26.7906
ZAR 18.2195 18.2270 1 8.22325 0.6634 0.6977 0.6806
GBP * 1.3197 1.3205 1 .32010 34.4716 36.2614 35.3665
JPY 150.5100 150.5200 1 50.51500 5.4809 5.7624 5.6217
ZMW/ZMK 22.8500 22.9000 2 2.87500 0.8321 0.8766 0.8544
BWP 0.0681 0.0792 0 .07365 1.7788 2.1748 1.9768
CHF 0.8135 0.8141 0 .81380 21.2492 22.3554 21.8023
MWK 1717.0200 1750.3200 1 ,733.67000 62.5272 67.0085 64.7679
AUD * 0.6435 0.6438 0 .64365 16.8087 17.6789 17.2438
SDR * 1.354910 1.354910 1 .35491 36.2988 36.2988 36.2988
MZN/MET 63.2500 64.5200 6 3.88500 2.3033 2.4700 2.3867
NOK 10.3257 10.3289 1 0.32730 0.3760 0.3954 0.3857
SEK 9.7902 9.7920 9 .79110 0.3565 0.3748 0.3657
CAD * 1.3853 1.3858 1 .38555 0.0504 0.0530 0.0517
EUR * 1.1423 1.1425 1 .14240 29.8378 31.3734 30.6056
CNY 7.2083 7.2094 7 .20885 0.2624 0.2760 0.2692
INR 87.2780 87.2880 8 7.28300 3.1783 3.3417 

In [13]:
import csv
FOLDER = Path("rbz_pdfs_may_sep_2025")
OUT_CSV = "usd_exchange_rates.csv"

records = []


for pdf_path in FOLDER.glob("*.pdf"):
    with pdfplumber.open(pdf_path) as pdf:
        text = "\n".join(page.extract_text() or "" for page in pdf.pages)

        date_match = re.search(r"(\d{1,2}\s+[A-Za-z]+\s+2025)", text)
        date = date_match.group(1) if date_match else None

        usd_line = None
        for line in text.splitlines():
            if line.strip().startswith("USD"):
                usd_line = line
                break

        if usd_line and date:
      
            nums = re.findall(r"-?\d+\.\d+", usd_line)
            
            if len(nums) >= 3:
                
                bid, ask, mid = nums[-3:]
                
                if mid.startswith('6.') and len(mid) > 2 and float(mid) < 10:
                    mid = '26' + mid[1:] 
                
                records.append({
                    "date": date,
                    "bid": float(bid),
                    "ask": float(ask),
                    "mid": float(mid),
                })
                print(f" {pdf_path.name} → {date} | {bid}/{ask}/{mid}")
            else:
                print(f"Not enough numbers found in USD line of {pdf_path.name}: {usd_line}")
        else:
            print(f"Could not extract USD data from {pdf_path.name}")


if records:
    df = pd.DataFrame(records)

    df["date"] = pd.to_datetime(df["date"], format="%d %B %Y")
    df = df.sort_values("date")

    with open(OUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["date", "bid", "ask", "mid"])
        writer.writeheader()
        for row in df.itertuples(index=False):
            writer.writerow({
                "date": row.date.strftime("%Y-%m-%d"),
                "bid": row.bid,
                "ask": row.ask,
                "mid": row.mid
            })

    print(f"\n Saved {len(df)} rows to {OUT_CSV}")
    print(df.head())
else:
    print("No data extracted")

CropBox missing from /Page, defaulting to MediaBox


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


 RATES_01_AUGUST_2025.pdf → 01 August 2025 | 26.1208/27.4604/26.7906


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


 RATES_01_JULY_2025.pdf → 01 July 2025 | 26.2764/27.6240/26.9502


...

 RATES_9_JULY_2025.pdf → 09 July 2025 | 26.2182/27.5628/26.8905


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


 RATES_9_JUNE_2025.pdf → 09 June 2025 | 26.2787/27.6263/26.9525


CropBox missing from /Page, defaulting to MediaBox


 RATES_9_MAY_2025.pdf → 09 May 2025 | 26.1691/27.5111/26.8401

 Saved 86 rows to usd_exchange_rates.csv
         date      bid      ask      mid
68 2025-05-02  26.1488  27.4898  26.8193
12 2025-05-05  26.1503  27.4913  26.8208
76 2025-05-06  26.1565  27.4979  26.8272
79 2025-05-07  26.1599  27.5015  26.8307
82 2025-05-08  26.1629  27.5045  26.8337
